# Bootstrap & Preflight — 08-Reporting.ipynb

**What this does**
- Verifies expected upstream notebooks have been executed and essential artefact folders exist.
- Prints clear guidance to run prerequisites if required files/folders are missing.

**Upstream prerequisites (recommended order)**
- `01-Setup_Preflight.ipynb`
- `02-Feature_Engineering.ipynb`
- `03-Feature_Selection.ipynb`
- `04-Baseline_and_BO.ipynb`

**Checks performed**
- Confirms `DATA_PATH` exists (from Section 0.1).
- Ensures `staging/` and `out/` directories exist when required downstream.
- Provides actionable instructions when a check fails.


In [1]:
# ======================================================
# Bootstrap & Preflight — 08-Reporting.ipynb
#   • Validates prerequisites and artefact folders
#   • Prints guidance if prerequisites are missing
# ======================================================
print(">>> Bootstrap & Preflight — 08-Reporting.ipynb")
required = ['01-Setup_Preflight.ipynb', '02-Feature_Engineering.ipynb', '03-Feature_Selection.ipynb', '04-Baseline_and_BO.ipynb']
print("[bootstrap] Recommended upstream notebooks:", required)

# Check DATA_PATH existence if declared
if 'DATA_PATH' in globals():
    from pathlib import Path as _P
    dp = _P(DATA_PATH)
    if not dp.exists():
        print(f"[bootstrap][warn] DATA_PATH not found: {dp} — please verify in 01-Setup_Preflight (Section 0.1).")

# Check staging/out directories
from pathlib import Path as _P
if 'STAGE_ROOT' in globals():
    sr = _P(STAGE_ROOT); 
    if not sr.exists():
        print(f"[bootstrap][warn] STAGE_ROOT does not exist: {sr}. Run 01-Setup_Preflight end-to-end first.")
if 'OUT_ROOT' in globals():
    oroot = _P(OUT_ROOT);
    if not oroot.exists():
        print(f"[bootstrap][warn] OUT_ROOT does not exist: {oroot}. It will be created as needed, but prior steps may be required.")

# Feature artefacts helpful for downstream
from pathlib import Path as _P
feat_dir = _P('staging') / 'feat'
if not feat_dir.exists():
    print("[bootstrap][hint] 'staging/feat' not found — this notebook can generate it (Sections 3.3/3.4), or run 03-Feature_Selection first.")
else:
    mi_file = feat_dir / 'mi_series.csv'
    if not mi_file.exists():
        print("[bootstrap][hint] MI series not found at 'staging/feat/mi_series.csv' — run Section 3.3 to generate.")

print("[bootstrap] Preflight checks complete. Proceed with this notebook if no critical warnings above.")


>>> Bootstrap & Preflight — 08-Reporting.ipynb
[bootstrap] Recommended upstream notebooks: ['01-Setup_Preflight.ipynb', '02-Feature_Engineering.ipynb', '03-Feature_Selection.ipynb', '04-Baseline_and_BO.ipynb']
[bootstrap] Preflight checks complete. Proceed with this notebook if no critical warnings above.


## Section 8 — Section

In [2]:
# =====================================================
# Section 8.1 — Report Manifest & README
#   • Imports pathlib/json (fixes NameError)
#   • Gathers report/model artefacts under OUT_ROOT
#   • Writes manifest.json + README.md
#   • (Optional) merges key metrics into metrics_summary.json
# =====================================================
print(">>> Section 8.1 — Report Manifest & README: start")

from pathlib import Path
import json, os

# Canonicals / dirs
OUT_ROOT = globals().get("OUT_ROOT", "out")
out_dir = Path(OUT_ROOT)
reports_dir = out_dir / "reports"
models_dir  = out_dir / "models"
metrics_dir = out_dir / "metrics"
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

# Collect artefacts (be generous in extensions)
report_exts = (".json", ".csv", ".png", ".pdf", ".html")
model_exts  = (".pkl", ".joblib", ".keras", ".onnx")

reports = sorted([p.name for p in reports_dir.iterdir() if p.suffix.lower() in report_exts])
models  = sorted([p.name for p in models_dir.iterdir()  if p.suffix.lower() in model_exts])

# Optional: pull best tree metric if persisted (from 04/06 as recommended)
best_tree_meta = {}
best_tree_path = metrics_dir / "best_tree.json"
if best_tree_path.exists():
    try:
        best_tree_meta = json.loads(best_tree_path.read_text())
    except Exception as e:
        print(f"[warn] Could not read {best_tree_path}: {e}")

# Optional: pick up CNN metric from globals or a file you may have written
cnn_val_acc = globals().get("CNN_VAL_ACC", None)
cnn_meta_path = metrics_dir / "cnn.json"
if (cnn_val_acc is None) and cnn_meta_path.exists():
    try:
        cnn_meta = json.loads(cnn_meta_path.read_text())
        cnn_val_acc = float(cnn_meta.get("val_acc", None))
    except Exception:
        pass

# Write manifest
manifest = {
    "reports": reports,
    "models": models,
    "metrics": {
        "best_tree": best_tree_meta if best_tree_meta else None,
        "cnn_val_acc": cnn_val_acc
    }
}
(manifest_path := out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"[ok] wrote manifest.json → {manifest_path}")

# Human-readable README
readme_text = """# IDS Final Report

This directory collects reports and trained model artefacts from the IDS training pipeline.

## Reports
- Evaluation assets (e.g., PR curves, confusion matrices), exported as JSON/CSV/PNG/PDF/HTML.

## Models
- Serialized estimators for deployment or inspection (e.g., LightGBM/XGBoost `.pkl`/`.joblib`, Keras `.keras`, ONNX `.onnx`).

## Metrics
- `metrics/best_tree.json`: best tree baseline/ensemble validation accuracy (from 04/06).
- `metrics/cnn.json` (optional): CNN validation accuracy (from 07).
- `manifest.json`: machine-readable index of artefacts + headline metrics.

"""
(readme_path := out_dir / "README.md").write_text(readme_text)
print(f"[ok] wrote README.md → {readme_path}")

# (Optional) compact summary for dashboards
summary = {
    "best_tree_val_acc": float(best_tree_meta.get("best_tree_val_acc", "nan")) if best_tree_meta else None,
    "best_tree_name": best_tree_meta.get("best_tree_name", None) if best_tree_meta else None,
    "cnn_val_acc": float(cnn_val_acc) if cnn_val_acc is not None else None,
    "n_reports": len(reports),
    "n_models": len(models),
}
(summary_path := metrics_dir / "metrics_summary.json").write_text(json.dumps(summary, indent=2))
print(f"[ok] wrote metrics summary → {summary_path}")

print(">>> Section 8.1 — complete")


>>> Section 8.1 — Report Manifest & README: start
[ok] wrote manifest.json → out/manifest.json
[ok] wrote README.md → out/README.md
[ok] wrote metrics summary → out/metrics/metrics_summary.json
>>> Section 8.1 — complete
